In [ ]:
# STRUCTURE:
# actual results, and initial stop (if never moved)
# no stop
# wide stop / max tolerable loss (20-40% range)
# 8, 10, 12.5, 15% stops
# 2, 2.5, 3 ATR stops
# maybe daily higher low and weekly higher low pivot stops, if I can figure that out, in a more advanced version

# ASSUMPTIONS:
# Entry is taken around the close of the day, so the first day is skipped when assessing R-multiples. Hence df.iloc[1:] syntax.
# On days where price gaps down below my stop, the stop is triggered at the open. Not always going to be the case
# No commissions or slippage on exit, so a round -1R loss if stop is hit intraday. Would like to fix in a future version.
# Intraday action is currently ignored. This may be an issue on days where my stop hits before a new high for the move. Will probably address with intraday price data and resampling. 

# FUTURE CONSIDERATIONS:
# Would I go about this process a different way with a larger dataset? Does pandas have a built in function, so I don't have to use the slower python loops?
# Instead of looping yfinance for every trade, should I store price data in a csv? Or would that be unnecessary? Depends on speed, data accuracy, etc. 
# Should I add some kind of drawdown calculation in a future version, so I can see what type of pullbacks I might expect and how viable wide stops actually are.
# While this isn't a consideration for v1, for future versions, I'd like to consider intraday stops, slippage, commissions,etc. I want to make this professional-grade.
# At some point I plan to test trailing stops on profitable trades to see what the most effective method there would be. E.g. pivot lows, moving average, atr, percentage trailing, etc.

# TASKS:
# Create pct_stop (consider whether catastrophe stop should be a row or column) and atr_stop functions.
# After creating pct_stop, change max_r_baseline to a catastrophe stop (e.g. 30-50%) as this is a more realistic baseline. Could keep no_stop as max_r_no_stop for additional value. 
# Should realised_r return current_r (from latest price) on open trades? What about exit_date and exit_price? Maybe leave exit_date and fill the others? Rename to last/final_price?
# Thinking it might be a good idea to use multiple functions for this. E.g. reading data, running simulation, outputting results.
# Could clean up the nested functions in stop_sim() by passing in dicts instead of repeating the same arguments. 

# ERRORS:
# pct_stop is almost identical to initial_stop; the loop phase is the same. I should make the loop its own function. 
# Every time I update something like false_negative_test, I then have to go and update the other stop functions, which is a bit of an annoyance. May need to do something about that.
# Minor inconsistency in functions and order of ticker/date return.

# EDGE CASES:
# Think about intraday entry and exit timing. E.g. what if my stop is below the low of the day, but I enter after the low of the day is set? Does it matter if entry is at the close?
# what if there's no data from df.iloc[1:]? 
# what should realised_r return if not stopped out? na value?
# what happens on a day if my stop hits before the high of the day? 

# REWORK:
# Fixed actual_outcome. Now, I need to rework the initial_stop function so that its code is reusable for other functions like pct_stop.
# Once I've done that, I'll then match the output of initial_stop and pct_stop to the format of actual_outcome.
# no_stop_max_r can be derived from this using the trade method's 1r value, if desired...
# Add realised_r (or available_r for open trades) and max_r back for each trade, and think about no_stop_max_r comparisons

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [7]:
# No stop function, returns max_price value in scenario with no stop loss

def no_stop(data, ticker, entry_date, entry_price, stop_price): 
    
    max_price = entry_price

    for row_name, row in data.iloc[1:].iterrows():

        if row['High'] > max_price:
            max_price = row['High']

    max_pct = ((max_price - entry_price) / entry_price) * 100

    return round(max_price, 2), round(max_pct, 2)

In [8]:
# False negative function, checks whether stop prevented trade from capturing sufficient portion of fat tail
def false_negative_test(max_pct, no_stop_max, pct_threshold, tail_threshold):

    if  (no_stop_max[1]) < 0.01:
        tail_pct = np.nan
    else: 
        tail_pct = round((max_pct / no_stop_max[1]) * 100, 2)

    if tail_pct > 100:
         tail_pct = 100
    
    if (no_stop_max[1] >= pct_threshold) and (tail_pct < tail_threshold):
            return True, tail_pct

    return False, tail_pct

In [9]:
# Actual trade result function

def actual_outcome(data, ticker, entry_date, entry_price, stop_price, exit_date, exit_price, no_stop_max, pct_threshold, tail_threshold):

    max_price = entry_price
    
    date = pd.to_datetime(entry_date) + pd.DateOffset(days=1) # Possible calendar date issue to be aware of here. We want trading days...
    
    if not pd.isna(exit_date):
        for row_name, row in data.loc[date:exit_date].iterrows(): # Ideally would use iloc[1: ...] as elsewhere...

            if row['High'] > max_price:
                max_price = row['High']
    else:
        for row_name, row in data.iloc[1:].iterrows():

            if row['High'] > max_price:
                max_price = row['High']

    max_pct = ((max_price - entry_price) / entry_price) *100

    false_negative = false_negative_test(max_pct, no_stop_max, pct_threshold, tail_threshold)

    return {'entry_date': entry_date, 
            'ticker': ticker, 
            'entry_price': entry_price,
            'stop_price': stop_price,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'max_pct': round(max_pct, 2),
            'no_stop_max_pct': no_stop_max[1],
            'tail_pct': round(false_negative[1], 2),
            'false_negative': false_negative[0],
            'stop_type': 'Actual Outcome'}

In [10]:
# Initial stop result function

def initial_stop(data, ticker, entry_date, entry_price, stop_price, no_stop_max, pct_threshold, tail_threshold):

    max_price = entry_price
    exit_date = np.nan
    exit_price = np.nan
    
    for row_name, row in data.iloc[1:].iterrows():

        if row['High'] > max_price:
            max_price = row['High']

        if stop_price >= row['Low']:
            if stop_price >= row['Open']:
                exit_price = row['Open']
                
            else:
                exit_price = stop_price
            
            exit_date = row_name.date()
            break

    if not pd.isna(exit_price):
        exit_price = round(exit_price, 2)

    max_pct = ((max_price - entry_price) / entry_price) *100

    false_negative = false_negative_test(max_pct, no_stop_max, pct_threshold, tail_threshold)

    return {'entry_date': entry_date, 
            'ticker': ticker, 
            'entry_price': entry_price,
            'stop_price': stop_price,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'max_pct': round(max_pct, 2),
            'no_stop_max_pct': no_stop_max[1],
            'tail_pct': round(false_negative[1], 2),
            'false_negative': false_negative[0],
            'stop_type': 'Initial Stop'}

In [8]:
# Percentage stop function - I think the logic here is largely the same as initial_stop minus the stop calculation. Probably doesn't need a separate function.

def pct_stop(data, ticker, entry_date, entry_price, stop_pct, max_r_baseline, r_threshold, tail_threshold):

    stop_price = round(entry_price * (1 - (stop_pct / 100)), 2)
    max_price = entry_price
    realised_r = np.nan
    exit_date = np.nan
    exit_price = np.nan

    for row_name, row in data.iloc[1:].iterrows():

        if row['High'] > max_price:
            max_price = row['High']

        if stop_price >= row['Low']:
            if stop_price >= row['Open']:
                realised_r = (row['Open'] - entry_price) / (entry_price - stop_price)
                exit_price = row['Open']
                
            else:
                realised_r = (stop_price - entry_price) / (entry_price - stop_price)
                exit_price = stop_price
            
            exit_date = row_name.date()
            break
    
    if not pd.isna(realised_r):
        realised_r = round(realised_r, 2)

    if not pd.isna(exit_price):
        exit_price = round(exit_price, 2)

    max_r = (max_price - entry_price) / (entry_price - stop_price)

    false_negative = false_negative_test(max_r, max_r_baseline, r_threshold, tail_threshold)

    return {'entry_date': entry_date, 
            'ticker': ticker, 
            'entry_price': entry_price, 
            'stop_price': stop_price, 
            'exit_date': exit_date,
            'exit_price': exit_price,
            'realised_r': realised_r, 
            'max_r': round(max_r, 2),
            'max_r_baseline': max_r_baseline,
            'false_negative': false_negative[0],
            'tail_capture_pct': false_negative[1],
            'stop_type': f'{stop_pct}_pct_stop'}


In [11]:
# Stop losses simulation function

def stop_sim(ticker, entry_date, entry_price, stop_price, exit_date, exit_price, pct_threshold, tail_threshold):
    
    data = yf.download(ticker, start=entry_date, multi_level_index=False, auto_adjust=True, progress=False)

    no_stop_max = no_stop(data, ticker, entry_date, entry_price, stop_price)

    sim_results = []
    sim_results.append(actual_outcome(data, ticker, entry_date, entry_price, stop_price, exit_date, exit_price, no_stop_max, pct_threshold, tail_threshold))
    sim_results.append(initial_stop(data, ticker, entry_date, entry_price, stop_price, no_stop_max, pct_threshold, tail_threshold))
    #sim_results.append(pct_stop(data, ticker, entry_date, entry_price, stop_pct, max_r_baseline, r_threshold, tail_threshold))

    return sim_results


In [12]:
# Reading trades CSV file, then using it as input for a list of dicts, which stores trade outputs.

trades = pd.read_csv('trades.csv')
stop_pct = 10
pct_threshold = 20
tail_threshold = 40

results = []

for row_name, row in trades.iterrows():
    results.extend(stop_sim(row['ticker'], row['entry_date'], row['entry_price'], row['stop_price'], row['exit_date'], row['exit_price'], pct_threshold, tail_threshold))


In [13]:
# List of nested dicts converted into dataframe

results_df = pd.DataFrame(results)
results_df

,entry_date,ticker,entry_price,stop_price,exit_date,exit_price,max_pct,no_stop_max_pct,tail_pct,false_negative,stop_type
0,2025-08-12,TSLA,340.84,320.15,2025-08-20,320.05,2.39,46.35,5.15,True,Actual Outcome
1,2025-08-12,TSLA,340.84,320.15,2025-08-20,320.15,2.39,46.35,5.15,True,Initial Stop
2,2025-08-26,STX,165.36,151.31,2025-11-21,229.72,79.36,340.25,23.32,True,Actual Outcome
3,2025-08-26,STX,165.36,151.31,NaN,NaN,340.25,340.25,100.00,False,Initial Stop
4,2025-08-26,CCL,31.89,29.34,2025-09-25,29.97,2.38,6.22,38.27,False,Actual Outcome
...,...,...,...,...,...,...,...,...,...,...,...
61,2026-04-08,AEIS,367.72,329.44,NaN,NaN,8.08,8.08,100.00,False,Initial Stop
62,2026-04-08,LITE,896.23,761.51,NaN,NaN,9.92,9.92,99.99,False,Actual Outcome
63,2026-04-08,LITE,896.23,761.51,NaN,NaN,9.92,9.92,99.99,False,Initial Stop
64,2026-04-08,STX,495.76,436.84,NaN,NaN,46.85,46.85,99.99,False,Actual Outcome


In [ ]:
# Using pandas functionality to output dataframe to CSV

results_df.to_csv('results.csv', index=False)